# py-spatialecotyper vs R SpatialEcoTyper 1.0.4 — pipeline parity

This notebook is the **reviewer-facing parity validation** for the port
`py-spatialecotyper`, a pure-Python re-implementation of the R package
[SpatialEcoTyper](https://github.com/digitalcytometry/spatialecotyper) 1.0.4
(Zhang *et al.*, *Nature* 2026, doi:10.1038/s41586-026-10452-4).

It follows the six-section schema in the rebuild kit's `NOTEBOOKS.md`:

1. Setup — the pre-registered gate (`data/manifest.yaml`) and the canonical fixture.
2. R reference run — the already-dumped R outputs plus a live R version probe.
3. Python candidate run — the pipeline executed live in this kernel.
4. Per-output parity — one subsection per `manifest.yaml::outputs[]` block.
5. Wall-clock comparison.
6. Verdict — the pre-registered gate with the measured value and PASS/FAIL per row.

**Every threshold below is read out of `data/manifest.yaml`**, which was committed
before any algorithmic Python was written and is read-only for this notebook.
Nothing is hard-coded, and nothing is rounded in the port's favour: one gate
(`similarity_network`) genuinely **fails**, and it is reported as a failure in
section 4.6 and again in the verdict table.

## 1. Setup

In [ ]:
# BLAS threads are pinned BEFORE numpy/scipy are imported, per the kit's
# engine/benchmark.py::lock_blas_threads convention, so the timings below are
# reproducible on a shared node.
import os
import sys

KIT = "/scratch/users/steorra/analysis/omicverse_dev/omicverse-rebuildr"
sys.path.insert(0, KIT)
from engine.benchmark import lock_blas_threads  # noqa: E402

lock_blas_threads(8)
print("BLAS thread pools pinned to:",
      {k: os.environ[k] for k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS",
                                  "MKL_NUM_THREADS")})

In [ ]:
import json
import re
import subprocess
import textwrap
import time

ROOT = os.getcwd() if os.path.exists("data/manifest.yaml") else os.path.abspath("..")
assert os.path.exists(os.path.join(ROOT, "data", "manifest.yaml")), ROOT
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "tests"))

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import yaml
from scipy.optimize import linear_sum_assignment
from scipy.spatial import procrustes
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import adjusted_rand_score

# tests/rio.py — the readers for the R dumps. Imported, never re-implemented.
import rio

import pyspatialecotyper as pse
from pyspatialecotyper import (core, metacells, network, preprocessing, rrandom,
                               seuratcompat, utils)
from pyspatialecotyper import nmf as nmf_mod
from pyspatialecotyper import stats as stats_mod
import pyspatialecotyper.integration as integ_mod

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "font.size": 9, "axes.titlesize": 10})
R_COLOR, PY_COLOR = "#0078d4", "#d13438"

print(f"py-spatialecotyper {pse.__version__}")
print(f"numpy {np.__version__} | scipy {sp.__version__ if hasattr(sp,'__version__') else ''}"
      f" | pandas {pd.__version__} | matplotlib {matplotlib.__version__}")
print(f"repo root: {ROOT}")

In [ ]:
# --- the pre-registered parity gate -----------------------------------------
with open(os.path.join(ROOT, "data", "manifest.yaml")) as fh:
    MANIFEST = yaml.safe_load(fh)

GATES = {o["name"]: o for o in MANIFEST["outputs"]}

print(f"package          : {MANIFEST['package']}")
print(f"upstream         : {MANIFEST['upstream']['name']} {MANIFEST['upstream']['version']}")
print(f"manifest created : {MANIFEST['created_at']} by {MANIFEST['created_by']}")
print()
print("HEADLINE ALGORITHM CLASS : "
      f"{MANIFEST['algorithm_class']}   (terminal scientific output: recurrent SE labels)")
print(f"HEADLINE THRESHOLD       : {MANIFEST['parity_threshold']}")
print()
print(f"{len(GATES)} pre-registered per-output gates; ALL must pass.")

In [ ]:
gate_table = pd.DataFrame(
    [{"output": o["name"], "R function": o["r_function"], "type": o["type"],
      "metric class": o["metric"], "threshold": o["threshold"]}
     for o in MANIFEST["outputs"]])
gate_table.index = np.arange(1, len(gate_table) + 1)
print("Pre-registered gate table (data/manifest.yaml::outputs[]):")
print(gate_table.to_string(max_colwidth=44))

In [ ]:
# --- canonical fixture + the R dump directories ------------------------------
CI = os.path.join(ROOT, "reference_out", "ci")       # 6000-cell spatial crop
FULL = os.path.join(ROOT, "reference_out", "full")   # all 27,907 cells
NMFD = os.path.join(ROOT, "reference_out", "nmf")    # NMF layer fixtures
INTEG = os.path.join(ROOT, "reference_out", "integ")  # real 2-sample integration
STATS = os.path.join(ROOT, "reference_out", "stats")  # downstream statistics

print("canonical fixture :", MANIFEST["fixture"]["path"],
      "->", MANIFEST["fixture"]["expected_shape"], "(genes x cells)")
print("second fixture    :", MANIFEST["second_fixture"]["path"],
      "(held out until the integration gate)")
print("CI sub-fixture    :", MANIFEST["ci_fixture"]["description"])
print()

PARAMS = rio.read_json(CI, "params")
print("R driver parameters replayed on both sides:")
print(json.dumps(PARAMS, indent=2))

In [ ]:
# The CI fixture, exactly as R saw it: R's own NormalizeData output + metadata.
norm = rio.read_sparse_raw(CI, "00_normdata").tocsc()
genes = rio.read_lines(CI, "00_normdata.rows")
cells = rio.read_lines(CI, "00_normdata.cols")
scmeta = rio.read_df(CI, "00_scmeta")

print(f"CI fixture       : {norm.shape[0]} genes x {norm.shape[1]} cells, "
      f"{norm.nnz} stored values")
print(f"cell types       : {sorted(scmeta['CellType'].unique())}")
print(f"regions          : {sorted(scmeta['Region'].astype(str).unique())}")

norm_full = None
full_cells = len(rio.read_lines(FULL, "00_normdata.cols"))
print(f"FULL fixture     : {len(rio.read_lines(FULL, '00_normdata.rows'))} genes "
      f"x {full_cells} cells (R dump only; not re-run here)")

In [ ]:
# Result accumulator for the verdict table in section 6.
RESULTS = []


def record(name, value, detail=""):
    """Score one manifest block against its pre-registered threshold."""
    g = GATES[name]
    cls, thr = g["metric"], float(g["threshold"])
    ok = (value < thr) if cls.startswith("deterministic") else (value >= thr)
    RESULTS.append({"output": name, "R function": g["r_function"],
                    "metric class": cls, "threshold": thr,
                    "measured": float(value), "PASS/FAIL": "PASS" if ok else "FAIL",
                    "detail": detail})
    arrow = "<" if cls.startswith("deterministic") else ">="
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:<26s} {cls:<22s} "
          f"measured = {value:.6g}  ({arrow} {thr:g})"
          + (f"   {detail}" if detail else ""))
    return ok


def flat(x):
    if sp.issparse(x):
        return np.asarray(x.todense()).ravel()
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return x.to_numpy().astype(float).ravel()
    return np.asarray(x, dtype=float).ravel()


def maxabs(a, b):
    a, b = flat(a), flat(b)
    assert a.shape == b.shape, f"shape mismatch {a.shape} vs {b.shape}"
    return float(np.max(np.abs(a - b)))


def show_deterministic(ref, cand, title, threshold, nshow=500):
    """The kit's `deterministic` visual: R vs Python overlay + max abs error."""
    r, c = flat(ref), flat(cand)
    err = np.abs(r - c)
    fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.1))
    idx = np.unique(np.linspace(0, len(r) - 1, min(nshow, len(r))).astype(int))
    axes[0].plot(np.arange(len(idx)), r[idx], lw=2.0, color=R_COLOR, label="R")
    axes[0].plot(np.arange(len(idx)), c[idx], lw=0.9, ls="--", color=PY_COLOR,
                 label="Python")
    axes[0].set_xlabel(f"evenly-spaced sample of {len(r)} values")
    axes[0].set_ylabel("value")
    axes[0].set_title("overlay (R solid, Python dashed)")
    axes[0].legend(loc="best", fontsize=8)

    axes[1].scatter(r, c, s=4, alpha=0.35, color="#333333")
    lo, hi = float(min(r.min(), c.min())), float(max(r.max(), c.max()))
    axes[1].plot([lo, hi], [lo, hi], color=PY_COLOR, lw=0.8, ls="--")
    axes[1].set_xlabel("R")
    axes[1].set_ylabel("Python")
    axes[1].set_title("element-wise")

    e = np.clip(err, 1e-20, None)
    axes[2].hist(np.log10(e), bins=50, color="#666666")
    axes[2].axvline(np.log10(threshold), color=PY_COLOR, ls="--", lw=1.4,
                    label=f"gate = {threshold:g}")
    axes[2].axvline(np.log10(max(err.max(), 1e-20)), color=R_COLOR, lw=1.4,
                    label=f"max abs err = {err.max():.3g}")
    axes[2].set_xlabel("log10 |R - Python|  (0 clipped to 1e-20)")
    axes[2].set_ylabel("count")
    axes[2].set_title("absolute error")
    axes[2].legend(loc="upper left", fontsize=7)
    fig.suptitle(title, y=1.04, fontsize=11)
    fig.tight_layout()
    return float(err.max())


def show_confusion(ref_labels, cand_labels, title, xlabel="Python", ylabel="R"):
    """The kit's `clustering` / `classification` visual: confusion-matrix heatmap."""
    rl = np.asarray([str(x) for x in ref_labels])
    cl = np.asarray([str(x) for x in cand_labels])
    rlev, clev = sorted(set(rl)), sorted(set(cl))
    m = pd.crosstab(pd.Series(rl, name=ylabel), pd.Series(cl, name=xlabel))
    fig, ax = plt.subplots(figsize=(0.55 * len(clev) + 3.4, 0.5 * len(rlev) + 2.6))
    im = ax.imshow(m.values, cmap="Blues", aspect="auto")
    ax.set_xticks(range(m.shape[1]))
    ax.set_xticklabels(m.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(m.shape[0]))
    ax.set_yticklabels(m.index, fontsize=7)
    ax.set_xlabel(f"{xlabel} label")
    ax.set_ylabel(f"{ylabel} label")
    ax.set_title(title)
    ax.grid(False)
    thr = m.values.max() / 2
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            v = m.values[i, j]
            if v:
                ax.text(j, i, str(v), ha="center", va="center", fontsize=6,
                        color="white" if v > thr else "black")
    fig.colorbar(im, ax=ax, shrink=0.8, label="cells / neighbourhoods")
    fig.tight_layout()
    return m


def show_ordinal(ref, cand, title, xlabel="R", ylabel="Python"):
    """The kit's `ordinal` visual: scatter + Pearson (and Spearman)."""
    r, c = flat(ref), flat(cand)
    mask = np.isfinite(r) & np.isfinite(c)
    pr = float(abs(pearsonr(r[mask], c[mask])[0]))
    sr = float(spearmanr(r[mask], c[mask])[0])
    fig, ax = plt.subplots(figsize=(4.4, 4.0))
    ax.scatter(r[mask], c[mask], s=12, alpha=0.55, color="#333333")
    lo, hi = float(min(r[mask].min(), c[mask].min())), float(max(r[mask].max(), c[mask].max()))
    ax.plot([lo, hi], [lo, hi], color=PY_COLOR, lw=0.9, ls="--", label="y = x")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{title}\nPearson = {pr:.6f}   Spearman = {sr:.6f}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    return pr, sr


def matched_factor_corr(ref, cand):
    """The port-local `factorization` class registered in manifest.yaml:
    Hungarian assignment on |Pearson|, statistic = mean matched |Pearson|."""
    k = ref.shape[1]
    c = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            a, b = ref[:, i], cand[:, j]
            c[i, j] = 0.0 if (np.std(a) == 0 or np.std(b) == 0) else abs(pearsonr(a, b)[0])
    r, cc = linear_sum_assignment(-c)
    return float(np.mean(c[r, cc])), c, (r, cc)


print("helpers ready")

## 2. R reference run

The R reference is **not** re-run here. `tests/r_reference_driver.R` takes 251.8 s
on the full fixture (and the two-sample integration driver plus the consensus-NMF
drivers take considerably longer), so the R side was executed once, stage by stage,
and dumped to `reference_out/`. This notebook loads those dumps through
`tests/rio.py` and compares against them.

What *is* run live below is a version probe: a one-line `Rscript -e` that prints the
R version and the versions of the three packages the reference depends on. The
recorded R wall-clock comes out of `reference_out/{ci,full}/timings.json`, which the
driver wrote with `difftime()` around each stage.

In [ ]:
R_BIN = "/scratch/users/steorra/env/CMAP/bin/Rscript"
R_ENV = dict(os.environ)
R_ENV["R_LIBS"] = ("/scratch/users/steorra/Rlibs_set:"
                   "/scratch/users/steorra/env/CMAP/lib/R/library:"
                   "/scratch/users/steorra/env/setref/lib/R/library")
R_ENV["TMPDIR"] = "/scratch/users/steorra/tmp"

R_PROBE = r'''
cat(R.version.string, "\n")
for (p in c("Seurat", "NMF", "SpatialEcoTyper", "Matrix", "RANN", "spdep", "irlba"))
  cat(sprintf("%-16s %s\n", p,
      tryCatch(as.character(packageVersion(p)), error = function(e) "not installed")))
'''

t0 = time.perf_counter()
proc = subprocess.run([R_BIN, "-e", R_PROBE], env=R_ENV, capture_output=True, text=True)
print(f"(live Rscript probe, {time.perf_counter() - t0:.2f} s)\n")
print(proc.stdout.strip())
if proc.returncode != 0:
    print("STDERR:", proc.stderr[-2000:])

In [ ]:
r_ci = rio.read_json(CI, "timings")
r_full = rio.read_json(FULL, "timings")
r_timings = pd.DataFrame({"R on CI crop (6000 cells) [s]": pd.Series(r_ci),
                          "R on full fixture (27907 cells) [s]": pd.Series(r_full)})
r_timings = r_timings.round(3)
print("Recorded R wall-clock (reference_out/*/timings.json):")
print(r_timings.to_string())
R_FULL_END2END = float(r_full["SpatialEcoTyper"])
R_CI_END2END = float(r_ci["SpatialEcoTyper"])
print(f"\nR SpatialEcoTyper end-to-end: {R_CI_END2END:.2f} s on the CI crop, "
      f"{R_FULL_END2END:.2f} s on the full fixture.")

## 3. Python candidate run

The Python pipeline is executed live in this kernel on the CI fixture (R's own
`NormalizeData` output as input, so the comparison starts from an identical
matrix). The full-fixture figure is the recorded one from `README.md`'s
performance table — re-running it here would add 84 s to every notebook build for
a number the port already measured.

In [ ]:
t0 = time.perf_counter()
res_py = core.spatial_ecotyper(
    norm, scmeta, gene_names=genes, radius=PARAMS["radius"],
    resolution=PARAMS["resolution"], nfeatures=PARAMS["nfeatures"],
    min_cts_per_region=PARAMS["min.cts.per.region"], npcs=PARAMS["npcs"],
    min_cells=PARAMS["min.cells"], min_features=PARAMS["min.features"],
    iterations=PARAMS["iterations"], k=PARAMS["k"], k_sn=PARAMS["k.sn"],
    verbose=False)
PY_CI_END2END = time.perf_counter() - t0

print(res_py)
print(f"\nPython SpatialEcoTyper end-to-end on the CI crop: {PY_CI_END2END:.2f} s")
print(f"R      SpatialEcoTyper end-to-end on the CI crop: {R_CI_END2END:.2f} s")
print(f"speed-up on the CI crop: {R_CI_END2END / PY_CI_END2END:.2f}x")
print()
print("spot metadata (head):")
print(res_py.spot_metadata.head())

In [ ]:
def readme_perf_table():
    """Parse the measured performance table out of README.md (repo file, not a
    typed-in constant) so the full-fixture numbers stay traceable."""
    txt = open(os.path.join(ROOT, "README.md")).read()
    rows = {}
    for m in re.finditer(r"^\|\s*(.+?)\s*\|\s*([\d.]+) s\s*\|\s*([\d.]+) s\s*\|", txt,
                         re.M):
        rows[m.group(1).strip().replace("`", "")] = (float(m.group(2)),
                                                    float(m.group(3)))
    return rows


PERF = readme_perf_table()
print("README.md performance table, parsed:")
for k, (rr, pp) in PERF.items():
    print(f"  {k:<48s} R {rr:7.1f} s   Python {pp:6.1f} s   {rr / pp:.1f}x")

FULL_R, FULL_PY = PERF["SpatialEcoTyper end-to-end"]
assert abs(FULL_R - R_FULL_END2END) < 0.1, "README disagrees with timings.json"
print(f"\ncross-check: README's R full-fixture figure {FULL_R} s matches "
      f"reference_out/full/timings.json ({R_FULL_END2END:.4f} s).")

## 4. Per-output parity

One subsection per `manifest.yaml::outputs[]` block, with the visual prescribed by
the block's parity class. Where a stage takes R's own upstream output as input, that
is stated — it isolates the stage instead of letting drift accumulate into it.

### 4.1 `preprocess_st` — `PreprocessST`

Pure filtering (genes in at least `min.cells` cells, cells with at least
`min.features` genes). Gated `deterministic-strict` at 1e-13: it must be bit-identical.

In [ ]:
exp, meta, g2, c2 = preprocessing.preprocess_st(
    norm, scmeta, min_cells=PARAMS["min.cells"],
    min_features=PARAMS["min.features"], genes=genes, cells=cells, verbose=False)
ref_exp = rio.read_sparse_raw(CI, "01_preprocess_expdat").tocsc()

print(f"R kept {ref_exp.shape[0]} genes x {ref_exp.shape[1]} cells; "
      f"Python kept {exp.shape[0]} x {exp.shape[1]}")
print("gene set identical:", g2 == rio.read_lines(CI, "01_preprocess_expdat.rows"))
print("cell set identical:", c2 == rio.read_lines(CI, "01_preprocess_expdat.cols"))

err = show_deterministic(ref_exp, exp, "PreprocessST — filtered expression matrix",
                         float(GATES["preprocess_st"]["threshold"]))
record("preprocess_st", err, f"{ref_exp.shape[0]}x{ref_exp.shape[1]} sparse")

### 4.2 `znorm` — `Znorm`

Weighted z-score of a gene x cell matrix, optionally within groups. Both the grouped
and ungrouped branches are compared against R's dump; the reported value is the worse
of the two. Gated `deterministic-standard` at 1e-8.

In [ ]:
V = rio.read_dense(NMFD, "nmf_V")
grp = np.array(["R1", "R2"] * (V.shape[1] // 2))[:V.shape[1]]
zn_g = preprocessing.znorm(V.to_numpy(), grp)
zn_n = preprocessing.znorm(V.to_numpy())
ref_g = rio.read_dense(NMFD, "nmf_znorm").to_numpy()
ref_n = rio.read_dense(NMFD, "nmf_znorm_nogroup").to_numpy()

e_g, e_n = maxabs(ref_g, zn_g), maxabs(ref_n, zn_n)
print(f"Znorm(groups=<2 groups>) max abs err : {e_g:.3g}")
print(f"Znorm(groups=None)       max abs err : {e_n:.3g}")

err = show_deterministic(ref_n, zn_n, "Znorm(groups=None) — weighted z-scores",
                         float(GATES["znorm"]["threshold"]))
record("znorm", max(e_g, e_n), f"worse of grouped/ungrouped, V = {V.shape}")

### 4.3 `rank_sparse` — `rankSparse`

Column-wise average-tie ranking of the non-zero entries of the fused graph, fed R's
own fused matrix so the ranking is isolated. Gated `deterministic-strict` at 1e-13;
this is one of the kernels the Acceleration loop rewrote (iteration 6), so bit
equality is the evidence that the rewrite was exact.

In [ ]:
fused_R = rio.read_sparse_raw(CI, "06_snf_fused").tocsr()
ref_rank = rio.read_sparse_raw(CI, "07_rank_sparse").tocsr()
py_rank = utils.rank_sparse(fused_R)

err = show_deterministic(ref_rank, py_rank, "rankSparse — column ranks of the fused graph",
                         float(GATES["rank_sparse"]["threshold"]))
record("rank_sparse", err, f"{ref_rank.shape[0]}x{ref_rank.shape[1]}, "
                           f"{ref_rank.nnz} stored")

### 4.4 `matrix_multiply` — `matrixMultiply`

The blocked sparse product. `matrixMultiply` splits the *second* operand by column,
which never reorders a summation index (proof in `MATH.md` section 3), so the answer
must not depend on `minibatch`. Compared against the unblocked product of the same
two R matrices at three block sizes.

In [ ]:
A_ = rio.read_sparse_raw(CI, "05_sn_B").tocsr()
B_ = rio.read_sparse_raw(CI, "05_sn_CD4T").tocsr()
ref_prod = np.asarray((A_ @ B_).todense())
mm_errs = {mb: maxabs(ref_prod, utils.matrix_multiply(A_, B_, minibatch=mb))
           for mb in (17, 64, 5000)}
print("max abs err vs the unblocked product, by minibatch:", mm_errs)

err = show_deterministic(ref_prod, utils.matrix_multiply(A_, B_, minibatch=17),
                         "matrixMultiply(minibatch=17) vs unblocked product",
                         float(GATES["matrix_multiply"]["threshold"]))
record("matrix_multiply", max(mm_errs.values()),
       "worst over minibatch in {17, 64, 5000}")

### 4.5 `spatial_metacells` — `GetSpatialMetacells`

Aggregates each cell's `k` nearest same-cell-type neighbours within `radius` into a
per-(neighbourhood, cell type) metacell profile: an unweighted, column-normalised KNN
weight matrix and one sparse matrix product. No decomposition on the path, so it is
gated `deterministic-standard` at 1e-8. Fed R's own filtered matrix and spot grid.

In [ ]:
ncmeta = core._spot_metadata(meta, PARAMS["grid.size"])
logexp = exp.copy()
if logexp.data.max() > 50:
    logexp.data = np.log1p(logexp.data)
mc, mc_cols = metacells.get_spatial_metacells(
    logexp, meta, spotCoord=ncmeta, k=PARAMS["k"], radius=PARAMS["radius"],
    gene_names=g2, verbose=False)
ref_mc = rio.read_sparse_raw(CI, "03_metacells").tocsc()

print(f"R metacells {ref_mc.shape}, Python {mc.shape}")
print("column set and order identical:",
      list(mc_cols) == rio.read_lines(CI, "03_metacells.cols"))
print(f"spatial neighbourhoods on the {PARAMS['grid.size']}-unit grid: "
      f"{len(ncmeta)} (R dumped {len(rio.read_df(CI, '02_ncmeta'))})")

err = show_deterministic(ref_mc, mc, "GetSpatialMetacells — metacell profiles",
                         float(GATES["spatial_metacells"]["threshold"]))
record("spatial_metacells", err, f"{mc.shape[0]} genes x {mc.shape[1]} "
                                 "(neighbourhood, cell type) columns")

### 4.6 `similarity_network` — `getSN` / `GetSNList`  — **this gate FAILS**

For each cell type, a `k`-nearest-neighbour graph over spatial neighbourhoods in PC
space, with inverse-distance weights. Both sides are fed **R's own PC embeddings**,
so this isolates the kNN + weighting step from anything upstream.

4 of the 9 cell types match at 1e-15. The other 5 do not, and the largest discrepancy
is `max abs = 0.11` on a 0-0.5 scale. The cause is a genuine tie: adjacent grid spots
that draw the same `k = 20` nearest cells of a rare cell type end up with *exactly
identical* metacell profiles, so their distances tie at the k-th-neighbour boundary,
and `RANN`'s ANN kd-tree and `scipy.spatial.cKDTree` keep different members of the tie
group. Both answers are correct k-NN sets; the tie is inherent to the data. This is
reported below as a **FAIL**, not rounded away and not excused by editing the manifest.

In [ ]:
r_emb = {}
for ct in rio.read_lines(CI, "04_emb_celltypes"):
    d = rio.read_dense(CI, f"04_emb_{ct}")
    r_emb[ct] = (d.to_numpy(), list(d.columns))

sn_py, sn_spots = network.get_sn_list(
    r_emb, npcs=PARAMS["npcs"], min_cts_per_region=PARAMS["min.cts.per.region"],
    k=PARAMS["k.sn"], verbose=False)

sn_rows = []
for ct in rio.read_lines(CI, "05_sn_celltypes"):
    ref = rio.read_sparse_raw(CI, f"05_sn_{ct}").tocsr()
    rc = rio.read_lines(CI, f"05_sn_{ct}.cols")
    w, names = sn_py[ct]
    pos = {s: i for i, s in enumerate(names)}
    w = sp.csr_matrix(w)[[pos[s] for s in rc]][:, [pos[s] for s in rc]]
    Aa, Bb = np.asarray(ref.todense()), np.asarray(w.todense())
    stored = (Aa != 0) | (Bb != 0)
    differing = np.abs(Aa - Bb) > 1e-12
    sn_rows.append({"cell type": ct, "spots": ref.shape[0],
                    "stored entries": int(stored.sum()),
                    "differing entries": int(differing.sum()),
                    "% identical": 100.0 * (1 - differing.sum() / stored.sum()),
                    "max abs err": float(np.abs(Aa - Bb).max())})
sn_df = pd.DataFrame(sn_rows).set_index("cell type")
print(sn_df.to_string(float_format=lambda v: f"{v:.6g}"))

SN_WORST = float(sn_df["max abs err"].max())
SN_TOTAL_STORED = int(sn_df["stored entries"].sum())
SN_TOTAL_DIFF = int(sn_df["differing entries"].sum())
SN_N_FAIL = int((sn_df["max abs err"] >= float(GATES["similarity_network"]["threshold"])).sum())
print(f"\n{SN_N_FAIL} of {len(sn_df)} cell types exceed the 1e-8 gate; "
      f"{len(sn_df) - SN_N_FAIL} are at the 1e-15 level.")
print(f"overall: {SN_TOTAL_DIFF} of {SN_TOTAL_STORED} stored entries differ "
      f"({100 * (1 - SN_TOTAL_DIFF / SN_TOTAL_STORED):.4f}% identical)")
print(f"worst single cell type (DC): "
      f"{sn_df.loc['DC', 'differing entries']:.0f} of "
      f"{sn_df.loc['DC', 'stored entries']:.0f} entries differ "
      f"({sn_df.loc['DC', '% identical']:.2f}% identical)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.4, 3.6))
x = np.arange(len(sn_df))
colors = [PY_COLOR if v >= float(GATES["similarity_network"]["threshold"]) else R_COLOR
          for v in sn_df["max abs err"]]
axes[0].bar(x, np.clip(sn_df["max abs err"], 1e-16, None), color=colors)
axes[0].set_yscale("log")
axes[0].axhline(float(GATES["similarity_network"]["threshold"]), color="black",
                ls="--", lw=1.2, label="pre-registered gate = 1e-8")
axes[0].set_xticks(x)
axes[0].set_xticklabels(sn_df.index, rotation=45, ha="right")
axes[0].set_ylabel("max abs |R - Python|")
axes[0].set_title("getSN per cell type (blue = clears the gate, red = fails)")
axes[0].legend(fontsize=8)

axes[1].bar(x, sn_df["% identical"], color=colors)
axes[1].set_ylim(99.0, 100.02)
axes[1].set_xticks(x)
axes[1].set_xticklabels(sn_df.index, rotation=45, ha="right")
axes[1].set_ylabel("% of stored entries identical")
axes[1].set_title("agreement on the stored support")
fig.tight_layout()

ct_worst = sn_df["max abs err"].idxmax()
ref = rio.read_sparse_raw(CI, f"05_sn_{ct_worst}").tocsr()
rc = rio.read_lines(CI, f"05_sn_{ct_worst}.cols")
w, names = sn_py[ct_worst]
pos = {s: i for i, s in enumerate(names)}
w = sp.csr_matrix(w)[[pos[s] for s in rc]][:, [pos[s] for s in rc]]
_ = show_deterministic(ref, w, f"getSN[{ct_worst}] — the worst cell type",
                       float(GATES["similarity_network"]["threshold"]))
record("similarity_network", SN_WORST,
       f"{SN_N_FAIL}/{len(sn_df)} cell types fail; "
       f"{SN_TOTAL_DIFF}/{SN_TOTAL_STORED} stored entries differ")

### 4.7 `snf_fused` — `SNF2`

The similarity-network-fusion diffusion, fed **R's own similarity-network list** so
the gate isolates the diffusion from the tie-breaking divergence in 4.6. This is the
stage the Acceleration loop's one bounded-epsilon rewrite touched (iteration 4,
`snf_sum_minus_self`), whose derived ceiling is 2.0e-14 (`MATH.md` section 2).

In [ ]:
cts_sn = rio.read_lines(CI, "05_sn_celltypes")
r_nets = [rio.read_sparse_raw(CI, f"05_sn_{ct}").tocsr() for ct in cts_sn]
t0 = time.perf_counter()
fused_py = network.snf2(r_nets, t=PARAMS["iterations"],
                        minibatch=PARAMS["minibatch"], verbose=False)
snf_t = time.perf_counter() - t0
print(f"SNF2 over {len(r_nets)} views, t = {PARAMS['iterations']} rounds: "
      f"Python {snf_t:.2f} s vs R {r_ci['SNF2']:.2f} s on the same crop")

err = show_deterministic(fused_R, fused_py, "SNF2 — fused similarity graph",
                         float(GATES["snf_fused"]["threshold"]))
print(f"derived (B) perturbation ceiling from MATH.md section 2: 2.0e-14; "
      f"measured {err:.3g}")
record("snf_fused", err, f"{len(r_nets)} views, t = {PARAMS['iterations']}")

### 4.8 `pc_embeddings` — `GetPCList`

Per-cell-type truncated SVD of the metacell matrix. R uses `irlba` (tol 1e-5), Python
a deterministic truncated SVD, so the gate is `embedding` (Procrustes similarity on
the top-20 PCs) rather than element-wise. The raw scatters can appear reflected or
rotated — that is exactly the ambiguity Procrustes quotients out.

In [ ]:
emb_py = network.get_pc_list(mc, mc_cols, g2, nfeatures=PARAMS["nfeatures"],
                             min_cells=PARAMS["min.cells"],
                             min_features=PARAMS["min.features"], verbose=False)
ct_list = rio.read_lines(CI, "04_emb_celltypes")
proc_sim = {}
fig, axes = plt.subplots(2, len(ct_list), figsize=(1.65 * len(ct_list), 4.0),
                         sharex=False, sharey=False)
for j, ct in enumerate(ct_list):
    ref = rio.read_dense(CI, f"04_emb_{ct}")
    e, spots = emb_py[ct]
    assert list(spots) == list(ref.columns)
    npc = min(ref.shape[0], e.shape[0], 20)
    _, _, disp = procrustes(ref.to_numpy()[:npc].T, e[:npc].T)
    proc_sim[ct] = 1 - disp
    axes[0, j].scatter(ref.to_numpy()[0], ref.to_numpy()[1], s=3, alpha=0.6,
                       color=R_COLOR)
    axes[0, j].set_title(f"{ct}\n1 - disp = {1 - disp:.6f}", fontsize=7)
    axes[1, j].scatter(e[0], e[1], s=3, alpha=0.6, color=PY_COLOR)
    for r_ in (0, 1):
        axes[r_, j].tick_params(labelsize=6)
    if j == 0:
        axes[0, j].set_ylabel("R\nPC2", fontsize=8)
        axes[1, j].set_ylabel("Python\nPC2", fontsize=8)
    axes[1, j].set_xlabel("PC1", fontsize=7)
fig.suptitle("GetPCList — first two PCs per cell type (R top, Python bottom)", y=1.02)
fig.tight_layout()

print(pd.Series(proc_sim, name="Procrustes similarity (1 - disparity), top 20 PCs")
      .to_string(float_format=lambda v: f"{v:.8f}"))
record("pc_embeddings", float(min(proc_sim.values())),
       f"worst of {len(proc_sim)} cell types")

### 4.9 `nmf_W` — `NMFGenerateW` / `NMFGenerateWList`

NMF factor columns carry no canonical order, so the manifest registers a port-local
`factorization` class: Hungarian assignment on the |Pearson| matrix, statistic = the
mean matched |Pearson|. `H` is held fixed by the upstream `NMFStrategy`, which makes
the KL objective convex in `W` (`MATH.md` section 1) — this is why an R function that
sets no seed is nonetheless reproducible.

In [ ]:
Fracs = rio.read_dense(NMFD, "nmf_Fracs")
refW = rio.read_dense(NMFD, "nmf_W_seed11")
W, wn, wses = nmf_mod.nmf_generate_w(
    Fracs.to_numpy(), V.to_numpy(), feature_names=list(V.index),
    se_names=list(Fracs.columns), scale=True, nfeature=300, nfeature_per_se=50)
common = [f for f in refW.index if f in set(wn)]
wi = {f: i for i, f in enumerate(wn)}
Wr, Wp = refW.loc[common].to_numpy(), W[[wi[f] for f in common]]

rvr = rio.read_json(NMFD, "nmf_W_R_vs_R")
print(f"R selected {refW.shape[0]} features, Python {W.shape[0]}; "
      f"Jaccard = {len(common) / len(set(refW.index) | set(wn)):.4f}")
print(f"R-vs-R control (two R runs from different RNG states): "
      f"feature Jaccard {rvr['jaccard']:.4f}, max|dW| {rvr['max_abs_diff_common']:.3g}, "
      f"cor {rvr['cor_common']:.6f}")

mW, cmatW, (rI, cI) = matched_factor_corr(Wr, Wp)
print(f"\nHungarian-matched mean |Pearson| = {mW:.8f}; "
      f"max abs diff on common features = {maxabs(Wr, Wp):.3g}")

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.0))
im = axes[0].imshow(cmatW, cmap="viridis", vmin=0, vmax=1)
axes[0].set_xlabel("Python factor")
axes[0].set_ylabel("R factor")
axes[0].set_title("|Pearson| between W columns")
axes[0].grid(False)
for i_, j_ in zip(rI, cI):
    axes[0].scatter([j_], [i_], marker="s", s=40, facecolors="none",
                    edgecolors="white", linewidths=1.2)
fig.colorbar(im, ax=axes[0], shrink=0.85)
axes[1].scatter(Wr[:, rI].ravel(), Wp[:, cI].ravel(), s=5, alpha=0.35, color="#333333")
lo, hi = float(Wr.min()), float(Wr.max())
axes[1].plot([lo, hi], [lo, hi], color=PY_COLOR, ls="--", lw=0.9)
axes[1].set_xlabel("R W (matched columns)")
axes[1].set_ylabel("Python W (matched columns)")
axes[1].set_title(f"matched W entries, mean |r| = {mW:.6f}")
fig.tight_layout()

record("nmf_W", mW, f"{len(common)} common features x {Wr.shape[1]} factors")

### 4.10 `nmf_H` — `NMFpredict` / `.nmf.predict`

The symmetric case: `W` is held fixed and only `H` is updated, so the KL objective is
convex in `H`. Both the single-chunk (`ncell.per.run = 5000`) and the chunked
(`ncell.per.run = 25`) branch of R's dump are compared; the reported statistic is the
worse of the two.

In [ ]:
H_stats = {}
for tag, chunk in (("nmf_H", 5000), ("nmf_H_chunked", 25)):
    refH = rio.read_dense(NMFD, tag).to_numpy()
    H, _, _ = nmf_mod.nmf_predict(refW.to_numpy(), list(refW.index),
                                  list(refW.columns), V.to_numpy(), list(V.index),
                                  list(V.columns), scale=False,
                                  ncell_per_run=chunk, sum2one=True)
    if refH.shape != H.shape:
        refH = refH.T
    mH, cmatH, (rIh, cIh) = matched_factor_corr(refH, H)
    H_stats[tag] = (mH, maxabs(refH, H), refH, H, cmatH, (rIh, cIh))
    print(f"{tag:<14s} (ncell.per.run = {chunk:>4d}): matched mean |Pearson| = "
          f"{mH:.8f}, max abs diff = {maxabs(refH, H):.3g}")
print(f"R-vs-R control on NMFpredict: max|dH| = "
      f"{rio.read_json(NMFD, 'nmf_H_R_vs_R')['max_abs_diff']:.3g}")

mH, _, refH, H, cmatH, (rIh, cIh) = H_stats["nmf_H"]
fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.0))
im = axes[0].imshow(cmatH, cmap="viridis", vmin=0, vmax=1)
axes[0].set_xlabel("Python factor")
axes[0].set_ylabel("R factor")
axes[0].set_title("|Pearson| between H rows/columns")
axes[0].grid(False)
for i_, j_ in zip(rIh, cIh):
    axes[0].scatter([j_], [i_], marker="s", s=40, facecolors="none",
                    edgecolors="white", linewidths=1.2)
fig.colorbar(im, ax=axes[0], shrink=0.85)
axes[1].scatter(refH[:, rIh].ravel(), H[:, cIh].ravel(), s=4, alpha=0.3,
                color="#333333")
lo, hi = float(refH.min()), float(refH.max())
axes[1].plot([lo, hi], [lo, hi], color=PY_COLOR, ls="--", lw=0.9)
axes[1].set_xlabel("R H (matched)")
axes[1].set_ylabel("Python H (matched)")
axes[1].set_title(f"matched H entries, mean |r| = {mH:.6f}")
fig.tight_layout()

record("nmf_H", min(v[0] for v in H_stats.values()),
       "worse of ncell.per.run in {5000, 25}")

### 4.11 `integrated_matrix` — `Integrate`

The cross-sample similarity matrix over spatial clusters, evaluated on the **held-out
real two-sample pair** (melanoma SKCM + colorectal CRC, `reference_out/integ`). The
CRC sample was withheld from every Equivalence-Agent iteration and used once, here.

In [ ]:
avg = rio.read_dense(INTEG, "integ_avgexprs")
ref_int = rio.read_dense(INTEG, "integ_integrated")
obj_int, spots_int = integ_mod.integrate(avg.to_numpy(), list(avg.index),
                                         list(avg.columns), nfeatures=300,
                                         min_features=10, seed=1)
dense_int = np.asarray(obj_int.todense())
pos = {s: i for i, s in enumerate(spots_int)}
perm = [pos[s] for s in ref_int.index]
print(f"avgexprs {avg.shape} (genes x spatial clusters, 2 samples pooled)")
print(f"integrated matrix {ref_int.shape}; spatial-cluster set identical: "
      f"{set(spots_int) == set(ref_int.index)}")
n_skcm = sum(1 for s in ref_int.index if s.startswith("SKCM"))
print(f"  SKCM spatial clusters: {n_skcm}; CRC: {len(ref_int.index) - n_skcm}")

err = show_deterministic(ref_int.to_numpy(), dense_int[perm][:, perm],
                         "Integrate — cross-sample similarity (melanoma + colorectal)",
                         float(GATES["integrated_matrix"]["threshold"]))
record("integrated_matrix", err,
       f"real 2-sample, {ref_int.shape[0]} spatial clusters")

### 4.12 `annotate_cells` — `AnnotateCells`

Transfers the spatial-neighbourhood SE label back onto individual cells by nearest
neighbourhood within `radius`. Deterministic given the neighbourhood labels, so both
sides are fed R's own neighbourhood labels. Gated `classification` at 0.99.

In [ ]:
ref_cells = rio.read_df(CI, "12_se_cells")
spot_json = rio.read_json(CI, "12_se_spot")
spot_meta = ncmeta.loc[list(spot_json["spot"])].copy()
spot_meta["SE"] = list(spot_json["SE"])
py_cells = core.annotate_cells(meta, spot_meta, radius=PARAMS["radius"], dropcell=True)

common_c = [c for c in ref_cells.index if c in set(py_cells.index)]
agree = float((ref_cells.loc[common_c, "SE"].astype(str).to_numpy()
               == py_cells.loc[common_c, "SE"].astype(str).to_numpy()).mean())
print(f"R annotated {len(ref_cells)} cells, Python {len(py_cells)}; "
      f"{len(common_c)} in common ({100 * len(common_c) / len(ref_cells):.2f}%)")
print(f"label agreement on the common cells: {agree:.6f}")

_ = show_confusion(ref_cells.loc[common_c, "SE"], py_cells.loc[common_c, "SE"],
                   f"AnnotateCells — per-cell SE labels (agreement = {agree:.4f})")
record("annotate_cells", agree, f"{len(common_c)} cells")

### 4.13 `se_labels_single_sample` — `SpatialEcoTyper`

**The headline end-to-end gate.** The full single-sample pipeline run live in section
3 — metacells, per-cell-type PCA, kNN similarity networks, SNF, rank transform, Seurat
PCA, SNN graph, Louvain — compared against R's spatial-neighbourhood labels by ARI.
Pre-registered at 0.85 (rather than the taxonomy default 0.95) with the written reason
recorded in the manifest: ten stages of accumulation, two of which (irlba truncated
SVD, Louvain modularity optimisation) are reproducible only up to their own
convergence tolerance.

In [ ]:
py_map = dict(zip(res_py.spot_metadata.index, res_py.spot_metadata["SE"]))
common_s = [s for s in spot_json["spot"] if s in py_map]
assert len(common_s) == len(spot_json["spot"]), "neighbourhood set differs from R"
r_map = dict(zip(spot_json["spot"], spot_json["SE"]))
r_lab = [r_map[s] for s in common_s]
p_lab = [py_map[s] for s in common_s]
ari = adjusted_rand_score(r_lab, p_lab)

print(f"spatial neighbourhoods: {len(common_s)} (identical set on both sides)")
print(f"R clusters: {len(set(r_lab))}, Python clusters: {len(set(p_lab))}")
print(f"ARI = {ari:.6f}")

cm = show_confusion(r_lab, p_lab,
                    f"SpatialEcoTyper — SE per spatial neighbourhood (ARI = {ari:.6f})")
print("\nconfusion matrix (R rows, Python columns):")
print(cm.to_string())
record("se_labels_single_sample", ari, f"{len(common_s)} spatial neighbourhoods")

### 4.14 `conserved_se_labels` — `nmfClustering` / `IntegrateSpatialEcoTyper`

Consensus NMF over `nrun.per.rank` random restarts, whose per-run seeds are
reproduced through the ported R Mersenne-Twister (`pyspatialecotyper.rrandom`). Two
instances are scored: the rank-8 clustering of the **real two-sample** integrated
matrix (the held-out gate) and the rank-4 fixture in `reference_out/nmf`. The
consensus matrices themselves are compared as a strictly stronger check than the label
ARI.

In [ ]:
ref_k8 = rio.read_json(INTEG, "integ_nmf_k8")
t0 = time.perf_counter()
res_k8 = nmf_mod.nmf_clustering(ref_int.to_numpy(), row_names=list(ref_int.index),
                                col_names=list(ref_int.columns), ranks=8,
                                nrun_per_rank=10, seed=1)
print(f"consensus NMF, rank 8, 10 restarts: {time.perf_counter() - t0:.1f} s")
ari_k8 = adjusted_rand_score(ref_k8["label"], res_k8["labels"])
cons_err_k8 = maxabs(rio.read_dense(INTEG, "integ_consensus_k8").to_numpy(),
                     res_k8["fits"]["K.8"]["consensus"])

S4 = rio.read_dense(NMFD, "nmf_integrated")
ref_k4 = rio.read_json(NMFD, "nmf_clustering_k4")
res_k4 = nmf_mod.nmf_clustering(S4.to_numpy(), row_names=list(S4.index),
                                col_names=list(S4.columns), ranks=4,
                                nrun_per_rank=10, seed=2024)
ari_k4 = adjusted_rand_score(ref_k4["label"], res_k4["labels"])
cons_err_k4 = maxabs(rio.read_dense(NMFD, "nmf_consensus_k4").to_numpy(),
                     res_k4["fits"]["K.4"]["consensus"])

print(f"rank 8, real 2-sample ({len(ref_k8['label'])} spatial clusters): "
      f"ARI = {ari_k8:.6f}, consensus max abs diff = {cons_err_k8:.3g}")
print(f"rank 4, nmf fixture   ({len(ref_k4['label'])} spatial clusters): "
      f"ARI = {ari_k4:.6f}, consensus max abs diff = {cons_err_k4:.3g}")
print(f"cophenetic (rank 8): R {ref_k8['cophenetic']:.6f}, "
      f"Python {float(res_k8['cophenetic']['Cophenetic'].iloc[0]):.6f}")
print(f"R-vs-R control on nmfClustering: identical labels = "
      f"{rio.read_json(NMFD, 'nmf_clustering_R_vs_R')['identical_labels']}")

_ = show_confusion(ref_k8["label"], res_k8["labels"],
                   f"nmfClustering rank 8, real 2-sample (ARI = {ari_k8:.6f})")
record("conserved_se_labels", min(ari_k8, ari_k4),
       f"worse of rank-8 real 2-sample and rank-4 fixture")

### 4.15 `compute_metrics` — `ComputeMetrics`

Recovery metrics (F1 / F2 / recall / precision) of a predicted SE assignment against
the reference one, on the downstream-statistics fixture (4000 cells). Six R dumps
cover the metric and grouping variants; the reported value is the worst.

In [ ]:
smeta = rio.read_df(STATS, "00_fixture")
for c in ("X", "Y", "Dist2Interface"):
    smeta[c] = smeta[c].astype(float)
counts_s = rio.read_dense(STATS, "00_counts")
SP = rio.read_json(STATS, "params")
print(f"statistics fixture: {smeta.shape[0]} cells x {smeta.shape[1]} metadata columns; "
      f"counts {counts_s.shape}")
print("R driver parameters:", json.dumps(SP))

metric_variants = [
    ("metrics_f1", dict(cell_type=None, sample="Sample", metric="F1")),
    ("metrics_f2", dict(cell_type=None, sample="Sample", metric="F2")),
    ("metrics_recall", dict(cell_type=None, sample="Sample", metric="recall")),
    ("metrics_precision", dict(cell_type=None, sample="Sample", metric="precision")),
    ("metrics_f1_ct", dict(cell_type="CellType", sample="Sample", metric="F1")),
    ("metrics_f1_nosample", dict(cell_type=None, sample=None, metric="F1")),
]
cm_errs = {}
for dump, kw in metric_variants:
    ref = rio.read_dense(STATS, dump)
    got = stats_mod.compute_metrics(smeta, se="SE", pred="cvPred", **kw)
    assert list(got.index) == list(ref.index) and list(got.columns) == list(ref.columns)
    cm_errs[dump] = maxabs(np.nan_to_num(ref.values.astype(float)),
                           np.nan_to_num(got.values.astype(float)))
print("\nmax abs error per variant:",
      {k: f"{v:.3g}" for k, v in cm_errs.items()})

ref = rio.read_dense(STATS, "metrics_f1")
got = stats_mod.compute_metrics(smeta, se="SE", pred="cvPred", cell_type=None,
                                sample="Sample", metric="F1")
err = show_deterministic(np.nan_to_num(ref.values.astype(float)),
                         np.nan_to_num(got.values.astype(float)),
                         "ComputeMetrics(metric='F1') — SE x SE recovery matrix",
                         float(GATES["compute_metrics"]["threshold"]))
record("compute_metrics", max(cm_errs.values()), "worst of 6 R dumps")

### 4.16 `coassociation_index` — `Coassociation(test = FALSE)`

The deterministic co-association kernel: for each pair of cell states, how often they
co-occur in the same SE across samples. 81 x 81 cell states on the statistics fixture.

In [ ]:
ref = rio.read_dense(STATS, "coassociation_index")
got = stats_mod.coassociation(smeta, sample="Sample", se="SEn", cell_type="CellType",
                              non_se="NonSE", test=False)
assert list(got.index) == list(ref.index)
print(f"co-association matrix: {ref.shape[0]} x {ref.shape[1]} cell states")
err = show_deterministic(ref.values.astype(float), got.values.astype(float),
                         "Coassociation(test = FALSE) — co-association index",
                         float(GATES["coassociation_index"]["threshold"]))
record("coassociation_index", err, f"{ref.shape[0]}x{ref.shape[1]} cell states")

### 4.17 `coassociation_pvals` — `CoassociationTest`

Permutation p-values over `nperm_coassoc` label shuffles. Reproducibility here rests
entirely on `pyspatialecotyper.rrandom`, the bit-identical port of R's Mersenne-Twister
and post-3.6 rejection sampler. Fed R's own co-association matrix, so the gate
isolates the permutation path. The `inference` class is Spearman on -log10 p together
with a top-k Jaccard.

In [ ]:
def named_series(refdir, name):
    j = rio.read_json(refdir, name)
    return pd.Series([np.nan if v is None else float(v) for v in j["value"]],
                     index=[str(s) for s in j["name"]])


mat_co = rio.read_dense(STATS, "coassociation_index")
ref_p = named_series(STATS, "coassociation_pvals")
ref_z = named_series(STATS, "coassociation_zscore")

rrandom.set_seed(int(SP["seed"]))
t0 = time.perf_counter()
got_p = stats_mod.coassociation_test(mat_co, nperm=int(SP["nperm_coassoc"]))
print(f"CoassociationTest, nperm = {SP['nperm_coassoc']}: "
      f"{time.perf_counter() - t0:.2f} s")

gp = got_p.reindex(ref_p.index).values
lr = -np.log10(np.clip(ref_p.values, 1e-300, None))
lp = -np.log10(np.clip(gp, 1e-300, None))
rho = float(spearmanr(lr, lp)[0])
z_err = maxabs(ref_z.values, got_p.attrs["Zscore"].reindex(ref_z.index).values)
print(f"tested pairs: {len(ref_p)}")
print(f"Spearman on -log10 p = {rho:.6f}; identical p-values: "
      f"{int((ref_p.values == gp).sum())}/{len(ref_p)}")
print(f"max abs difference on the underlying Z-scores = {z_err:.3g}")

fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.scatter(lr, lp, s=40, alpha=0.7, color="#333333")
lo, hi = float(min(lr.min(), lp.min())), float(max(lr.max(), lp.max()))
ax.plot([lo, hi], [lo, hi], color=PY_COLOR, ls="--", lw=0.9, label="y = x")
ax.set_xlabel("R  -log10(p)")
ax.set_ylabel("Python  -log10(p)")
ax.set_title(f"CoassociationTest permutation p-values\nSpearman = {rho:.6f}")
ax.legend(fontsize=8)
fig.tight_layout()
record("coassociation_pvals", rho, f"{len(ref_p)} pairs, "
                                   f"nperm = {SP['nperm_coassoc']}, Z max|d| = {z_err:.2g}")

### 4.18 `colocalization_index` — `Colocalization`

Permutation Z-scores for spatial co-localisation of cell states. R's own loop is only
reproducible at `ncores = 1` (`mclapply` forks and each child reseeds from its PID),
so the reference was generated at `ncores = 1` and the port consumes the same RNG
stream sequentially. Gated `ordinal` (Pearson) at 0.99.

In [ ]:
ref_coloc = rio.read_dense(STATS, "colocalization_index")
rrandom.set_seed(int(SP["seed"]))
t0 = time.perf_counter()
got_coloc = stats_mod.colocalization(
    smeta, coords=("X", "Y"), se="SE", cell_type="CellType", radius=50,
    k=int(SP["k_coloc"]), min_cell=10, nperm=int(SP["nperm_coloc"]), test=True,
    ncores=1)
PY_COLOC_S = time.perf_counter() - t0
assert list(got_coloc["ColocIndex"].index) == list(ref_coloc.index)
print(f"Colocalization, {ref_coloc.shape[0]} cell states, "
      f"nperm = {SP['nperm_coloc']}: Python {PY_COLOC_S:.2f} s")
print(f"max abs difference on the Z-score matrix = "
      f"{maxabs(ref_coloc.values.astype(float), got_coloc['ColocIndex'].values.astype(float)):.3g}")
rvr_c = rio.read_json(STATS, "colocalization_rvr")
print("R-vs-R control (ncores > 1, same set.seed):", json.dumps(rvr_c))

pr, sr = show_ordinal(ref_coloc.values.astype(float),
                      got_coloc["ColocIndex"].values.astype(float),
                      "Colocalization — permutation Z-scores",
                      xlabel="R Z-score", ylabel="Python Z-score")
record("colocalization_index", pr,
       f"{ref_coloc.shape[0]}x{ref_coloc.shape[1]}, Spearman = {sr:.6f}")

### 4.19 `normalized_moran_i` — `ComputeNormalizedMoranI`

Moran's I of each SE's spatial distribution, normalised by a permutation null. R's
reference came from `spdep` 1.4.2; the port evaluates the closed form that
`style = "W"` weights collapse to (`MATH.md` section 4), so it needs no
`sf`/`GEOS`/`PROJ` stack. Gated `ordinal` at 0.95.

In [ ]:
ref_moran = named_series(STATS, "normalized_moran_i")
rrandom.set_seed(int(SP["seed"]))
t0 = time.perf_counter()
got_moran = stats_mod.compute_normalized_moran_i(
    smeta, coords=("X", "Y"), se="SE", cell_type="CellType",
    nperm=int(SP["nperm_moran"]), k=int(SP["k_moran"]), ncores=1)
print(f"ComputeNormalizedMoranI, nperm = {SP['nperm_moran']}, k = {SP['k_moran']}: "
      f"{time.perf_counter() - t0:.2f} s")
lm = rio.read_json(STATS, "moran_listw_meta")
print(f"spdep listw check: S0 = {lm['S0']}, n = {lm['n']} "
      f"(style='W' implies S0 == n)")
print(pd.DataFrame({"R": ref_moran, "Python": got_moran.reindex(ref_moran.index)})
      .to_string(float_format=lambda v: f"{v:.10f}"))
print(f"max abs difference = {maxabs(ref_moran.values, got_moran.reindex(ref_moran.index).values):.3g}")

pr, sr = show_ordinal(ref_moran.values, got_moran.reindex(ref_moran.index).values,
                      "ComputeNormalizedMoranI — normalised Z-scores",
                      xlabel="R (spdep 1.4.2)", ylabel="Python")
record("normalized_moran_i", pr, f"{len(ref_moran)} SEs")

### 4.20 `se_abundance_by_sn` — `ComputeSEAbundanceBySN` / `SmoothSEAbundances`

Per-spot SE abundance from a KNN weight matrix, and its spatial smoothing. The
returned data frame carries the spot coordinates in its first two columns; those are
grid coordinates, not payload, and are compared separately. Three R dumps are scored:
the raw 0/1 KNN weights, the aggregation, and the smoothed abundances.

In [ ]:
coords = smeta[["X", "Y"]]
knn_errs = {}
for dump, self_flag in (("knn_weights_self", True), ("knn_weights_noself", False)):
    refw = rio.read_sparse_raw(STATS, dump).toarray()
    gotw = stats_mod.build_knn_weights(ref_coords=coords, k=20, radius=40,
                                       include_self=self_flag).toarray()
    knn_errs[dump] = maxabs(refw, gotw)

ref_ab = rio.read_df(STATS, "se_abundance_by_sn")
got_ab = stats_mod.compute_se_abundance_by_sn(smeta, spot_coords=None, radius=50,
                                              grid_size=50, x="X", y="Y", se="SE",
                                              min_cells=5)
assert list(got_ab.index) == list(ref_ab.index)
pay_ref = ref_ab.iloc[:, 2:].values.astype(float)
pay_got = got_ab.iloc[:, 2:].values.astype(float)
xy_err = maxabs(ref_ab.iloc[:, :2].values.astype(float),
                got_ab.iloc[:, :2].values.astype(float))

ref_sm = rio.read_df(STATS, "smooth_se_abundances")
got_sm = stats_mod.smooth_se_abundances(got_ab.iloc[:, 2:], got_ab[["X", "Y"]], k=7,
                                        x="X", y="Y", include_self=True,
                                        min_neighbors=3)
sm_err = maxabs(ref_sm.values.astype(float), got_sm.values.astype(float))

print("buildKNNWeights max abs err:", {k: f"{v:.3g}" for k, v in knn_errs.items()})
print(f"ComputeSEAbundanceBySN: {ref_ab.shape[0]} spots x "
      f"{pay_ref.shape[1]} SE columns; payload max abs err = "
      f"{maxabs(pay_ref, pay_got):.3g}")
print(f"  (spot X/Y grid coordinates differ by {xy_err:.3g} — these are coordinates, "
      "not payload)")
print(f"SmoothSEAbundances max abs err = {sm_err:.3g}")

err = show_deterministic(pay_ref, pay_got,
                         "ComputeSEAbundanceBySN — SE abundance per spot",
                         float(GATES["se_abundance_by_sn"]["threshold"]))
record("se_abundance_by_sn",
       max(maxabs(pay_ref, pay_got), sm_err, max(knn_errs.values())),
       "worst of KNN weights, abundances (payload), smoothing")

### 4.21 Remaining manifest blocks

Three further blocks have R dumps and are scored below for completeness:
`infer_ncells`, `partition_tissue` and `colocalization_meta`.

In [ ]:
inf_errs = {}
for tag, av in (("5", 5), ("1p5", 1.5), ("12", 12)):
    r_ = np.asarray(rio.read_json(STATS, f"infer_ncells_{tag}")["ncells"], dtype=np.int64)
    g_ = stats_mod.infer_ncells(counts_s.values, avg_number=av)
    inf_errs[f"avg_number={av}"] = maxabs(r_.astype(float), g_.astype(float))
print("InferNCells max abs err:", inf_errs)
record("infer_ncells", max(inf_errs.values()), "avg_number in {5, 1.5, 12}")

ref_pt = rio.read_json(STATS, "partition_tissue")
got_pt = stats_mod.partition_tissue(smeta, nrow=3, ncol=5, x="X", y="Y")
pt_agree = float(np.mean(np.array([str(v) for v in ref_pt["partition"]])
                         == got_pt["Partition"].astype(str).values))
print(f"PartitionTissue agreement on {len(got_pt)} cells: {pt_agree:.6f}")
record("partition_tissue", pt_agree, f"{len(got_pt)} cells, 3 x 5 grid")

meta_inputs = []
for i in (1, 2, 3):
    idx = rio.read_dense(STATS, f"meta_input_index_S{i}")
    pv = named_series(STATS, f"meta_input_pval_S{i}")
    pv.attrs["Zscore"] = named_series(STATS, f"meta_input_zscore_S{i}")
    meta_inputs.append({"ColocIndex": idx, "Pval": pv})
meta_prs = []
for tag, kw in (("", dict(cap=5, min_samples=1)), ("_cap2", dict(cap=2, min_samples=2))):
    ref_mi = rio.read_dense(STATS, f"colocalization_meta_index{tag}")
    got_mi = stats_mod.colocalization_meta_analysis(meta_inputs, **kw)
    p_ = float(abs(pearsonr(ref_mi.values.astype(float).ravel(),
                            got_mi["MetaColocIndex"].values.astype(float).ravel())[0]))
    meta_prs.append(p_)
    print(f"ColocalizationMetaAnalysis({kw}): Pearson = {p_:.8f}, max abs err = "
          f"{maxabs(ref_mi.values.astype(float), got_mi['MetaColocIndex'].values.astype(float)):.3g}")
record("colocalization_meta", min(meta_prs), "worse of cap=5 and cap=2 branches")

### 4.22 Manifest blocks with no R dump

Three manifest blocks are **not** scored in this notebook, and are listed here rather
than silently skipped:

* **`aggregate_recover_models`** (`AggregateRecoverModels`) — no dump in
  `reference_out/`. The R function aggregates the bundled MERSCOPE recovery models
  (`inst/extdata/*.rds`), which upstream's licence does not allow redistributing, so
  there is no reference input to feed both sides.
* **`deconvolute_se`** (`DeconvoluteSE`) — same reason: the published `W` it
  deconvolutes against lives in the same non-redistributable `.rds` bundle.
* **`recover_se_labels`** (`RecoverSE`) — same reason; `recover_se` requires an
  explicit `Ws` argument in this port because the default-model branch is unreachable
  without those files.

These three blocks therefore appear below as `no dump`, with no measured value, and
are excluded from the pass count rather than counted as passes. The recovery path is
still exercised for shape/API correctness by the package's own tests; it simply has
no R-vs-Python numeric gate.

In [ ]:
NO_DUMP = ["aggregate_recover_models", "deconvolute_se", "recover_se_labels"]
for n in NO_DUMP:
    g = GATES[n]
    print(f"  [no dump] {n:<26s} {g['metric']:<22s} gate {g['threshold']}  "
          f"({g['r_function']})")
scored = {r["output"] for r in RESULTS}
unscored = [n for n in GATES if n not in scored]
print(f"\nscored {len(scored)} of {len(GATES)} manifest blocks; "
      f"unscored: {sorted(unscored)}")

## 5. Wall-clock comparison

In [ ]:
COLOC_R, COLOC_PY_README = PERF["Colocalization (same seed, same nperm)"]

bars = pd.DataFrame({
    "R": [FULL_R, COLOC_R, R_CI_END2END],
    "Python": [FULL_PY, PY_COLOC_S, PY_CI_END2END],
}, index=[f"SpatialEcoTyper end-to-end\nfull fixture ({full_cells} cells)",
          f"Colocalization\n{smeta.shape[0]} cells, nperm = {SP['nperm_coloc']}",
          f"SpatialEcoTyper end-to-end\nCI crop ({norm.shape[1]} cells)"])
bars["speed-up"] = bars["R"] / bars["Python"]
print(bars.rename(index=lambda s: s.replace("\n", " "))
      .to_string(float_format=lambda v: f"{v:.2f}"))
print(f"\n(Colocalization Python figure measured live in section 4.18: "
      f"{PY_COLOC_S:.2f} s; README records {COLOC_PY_README} s.)")

fig, ax = plt.subplots(figsize=(8.6, 4.0))
x = np.arange(len(bars))
w = 0.36
b1 = ax.bar(x - w / 2, bars["R"], w, color=R_COLOR, label="R 4.4.3 / SpatialEcoTyper 1.0.4")
b2 = ax.bar(x + w / 2, bars["Python"], w, color=PY_COLOR,
            label=f"py-spatialecotyper {pse.__version__}")
for rect, v in zip(b1, bars["R"]):
    ax.text(rect.get_x() + rect.get_width() / 2, v, f"{v:.1f} s", ha="center",
            va="bottom", fontsize=8)
for rect, v, s in zip(b2, bars["Python"], bars["speed-up"]):
    ax.text(rect.get_x() + rect.get_width() / 2, v, f"{v:.1f} s\n{s:.1f}x",
            ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(bars.index, fontsize=8)
ax.set_ylabel("wall-clock (s)")
ax.set_ylim(0, bars[["R", "Python"]].values.max() * 1.25)
ax.set_title("Wall-clock: R reference vs Python port")
ax.legend(fontsize=8)
fig.tight_layout()

The port carries a single (B) bounded-epsilon rewrite (Acceleration iteration 4,
derived ceiling 2.0e-14, `MATH.md` section 2); every other accepted rewrite is exact,
so there is no separate "acceleration" bar to draw — the Python bars above already
include all accepted rewrites. The per-iteration trajectory is in
[`examples/evolution.ipynb`](evolution.ipynb) and [`ITERATION_LOG.md`](../ITERATION_LOG.md).

## 6. Verdict

In [ ]:
verdict = pd.DataFrame(RESULTS)
order = [o["name"] for o in MANIFEST["outputs"]]
verdict["_o"] = verdict["output"].map({n: i for i, n in enumerate(order)})
verdict = verdict.sort_values("_o").drop(columns="_o").reset_index(drop=True)
verdict.index = np.arange(1, len(verdict) + 1)


def fmt(v):
    return f"{v:.6g}"


print("PRE-REGISTERED GATE (data/manifest.yaml) vs MEASURED")
print("=" * 118)
print(verdict.assign(measured=verdict["measured"].map(fmt),
                     threshold=verdict["threshold"].map(fmt))
      .to_string(max_colwidth=52))
print("=" * 118)
for n in NO_DUMP:
    g = GATES[n]
    print(f"    {n:<26s} {g['metric']:<22s} threshold {g['threshold']:<8} "
          f"measured: no dump (see section 4.22)")

n_pass = int((verdict["PASS/FAIL"] == "PASS").sum())
n_fail = int((verdict["PASS/FAIL"] == "FAIL").sum())
print(f"\n{n_pass} PASS, {n_fail} FAIL out of {len(verdict)} scored blocks "
      f"({len(NO_DUMP)} further blocks have no dump).")

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 0.34 * len(verdict) + 1.4))
y = np.arange(len(verdict))[::-1]
cols = [R_COLOR if s == "PASS" else PY_COLOR for s in verdict["PASS/FAIL"]]
# Normalised distance to the gate: >0 means inside the gate.
norm_margin = []
for _, row in verdict.iterrows():
    if row["metric class"].startswith("deterministic"):
        norm_margin.append(np.log10(row["threshold"]) -
                           np.log10(max(row["measured"], 1e-18)))
    else:
        norm_margin.append((row["measured"] - row["threshold"]) * 20)
ax.barh(y, norm_margin, color=cols)
ax.axvline(0, color="black", lw=1.2)
ax.set_yticks(y)
ax.set_yticklabels([f"{r['output']}  ({r['metric class']})"
                    for _, r in verdict.iterrows()], fontsize=7)
ax.set_xlabel("margin inside the pre-registered gate\n"
              "(deterministic blocks: orders of magnitude below the threshold; "
              "others: 20 x (metric - threshold))")
ax.set_title("Margin against the pre-registered gate (right of 0 = PASS)")
fig.tight_layout()

In [ ]:
failing = verdict[verdict["PASS/FAIL"] == "FAIL"]
lines = []
if len(failing) == 0:
    FINAL = ("PASS - all outputs cleared the pre-registered gate in "
             "data/manifest.yaml.")
else:
    parts = "; ".join(
        f"{r['output']} ({r['metric class']}) measured {r['measured']:.6g} "
        f"against a threshold of {r['threshold']:g}"
        for _, r in failing.iterrows())
    FINAL = (f"FAIL - {n_pass} of {len(verdict)} scored outputs cleared the "
             f"pre-registered gate; {n_fail} did not: {parts}.")

print(textwrap.fill(FINAL, 100))
print()
print(textwrap.fill(
    "Reason for the one failure: getSN selects k nearest spatial neighbourhoods in PC "
    "space. In 5 of the 9 cell types the canonical fixture contains neighbourhoods "
    "whose cell-type-specific metacell profiles are exactly identical - adjacent grid "
    "spots drawing the same k = 20 nearest cells of a rare type - so their distances "
    "tie at the k-th-neighbour boundary. RANN's ANN kd-tree and scipy.spatial.cKDTree "
    "resolve that tie differently, and both resolutions are valid k-NN sets. "
    f"Measured effect: {SN_TOTAL_DIFF} of {SN_TOTAL_STORED} stored entries differ "
    f"across all nine networks "
    f"({100 * (1 - SN_TOTAL_DIFF / SN_TOTAL_STORED):.4f}% identical), the worst "
    f"single cell type being DC at "
    f"{sn_df.loc['DC', 'differing entries']:.0f}/{sn_df.loc['DC', 'stored entries']:.0f} "
    f"({sn_df.loc['DC', '% identical']:.2f}% identical), and the largest discrepancy "
    f"is max abs {SN_WORST:.4g} on a 0-0.5 weight scale. Four of the nine cell types "
    "match at the 1e-15 level. The manifest was not edited to accommodate this, and "
    "the failure is carried into README.md's 'Known divergences' section.", 100))
print()
print(textwrap.fill(
    "Downstream impact, measured: SNF2 fed R's own network list matches at "
    f"{verdict.set_index('output').loc['snf_fused', 'measured']:.3g}, and the "
    "end-to-end clustering gate se_labels_single_sample still clears its "
    f"pre-registered 0.85 threshold at ARI "
    f"{verdict.set_index('output').loc['se_labels_single_sample', 'measured']:.6f}.",
    100))